# Chapter 13 — Compilation: Which Assumption Stopped Holding?

**Book alignment:** PyTorch From First Principles, Chapter 13

**Question this notebook isolates:** Does a compiled tiny function stay numerically correct (eager agreement) while capture evidence (graph count, break reasons) — not the correct numbers alone — determines whether compilation is stable and worth it? (torch.compile usage minimal and guarded; eager fallback on failure.)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__)

## 1 — Correctness first: eager bisection, then compiled agreement

If eager is wrong the compiler is a distraction. A healthy compile agrees with eager at a stated tolerance on a tiny function.

In [ ]:
def tiny_fn(x):
    return torch.cos(torch.sin(x) @ x.T + 1.0).sum(dim=-1)

x = torch.randn(4, 8)
eager_out = tiny_fn(x)
print('eager ok:', tuple(eager_out.shape), bool(torch.isfinite(eager_out).all()))

compiled_out, compile_note = None, 'skipped'
try:
    cfn = torch.compile(tiny_fn, fullgraph=False)
    compiled_out = cfn(x)
    compile_note = 'compiled'
except Exception as e:
    compiled_out = eager_out
    compile_note = f'fallback: {type(e).__name__}'
print(compile_note, 'max diff:', float((eager_out - compiled_out).abs().max()))

In [ ]:
assert tuple(eager_out.shape) == (4,)
assert bool(torch.isfinite(eager_out).all())
assert float((eager_out - compiled_out).abs().max()) < 1e-4
print('RUNS + CORRECT verified')

## 2 — Capture: a data-dependent branch fragments the graph

`torch._dynamo.explain` counts graphs and breaks; `fullgraph=True` turns the silent break into an error. A print-style `.item()` branch is the canonical break.

In [ ]:
def clean_fn(x):
    return torch.cos(torch.sin(x) @ x.T).sum()

def break_fn(x):
    y = torch.sin(x) @ x.T
    if x.sum().item() > 0:
        y = y * 2
    return torch.cos(y).sum()

info = {}
try:
    e_clean = torch._dynamo.explain(clean_fn)(x)
    e_break = torch._dynamo.explain(break_fn)(x)
    info['clean'] = (e_clean.graph_count, e_clean.graph_break_count)
    info['break'] = (e_break.graph_count, e_break.graph_break_count)
    print('clean graphs/breaks:', info['clean'])
    print('break graphs/breaks:', info['break'])
    print('break reasons:', list(e_break.break_reasons)) 
except Exception as e:
    info['error'] = type(e).__name__
    print('explain unavailable:', info['error'])

fullgraph_raises = None
try:
    torch.compile(break_fn, fullgraph=True)(x)
    fullgraph_raises = False
except Exception:
    fullgraph_raises = True
print('fullgraph=True raises on break:', fullgraph_raises)

In [ ]:
assert fullgraph_raises is True
if 'clean' in info:
    assert info['clean'][1] == 0
    assert info['break'][0] >= 2 or info['break'][1] >= 1
print('CAPTURED evidence verified')

## 3 — Stability sketch: one compiled artifact must serve varying shapes correctly

Specializing per shape recompiles per shape; the stability question is whether reuse holds. Here we check the portable half: compiled outputs stay correct across shapes (guards/reuse discussed in the book).

In [ ]:
def flex_fn(x):
    return (torch.tanh(x) * 2.0 + 0.5).sum(dim=-1)

try:
    cflex = torch.compile(flex_fn, fullgraph=False, dynamic=None)
    diffs = []
    for T in [8, 12, 8]:
        xt = torch.randn(2, T)
        d = float((cflex(xt) - flex_fn(xt)).abs().max())
        diffs.append(d)
        print(f'T={T} max diff={d:.2e}')
except Exception as e:
    diffs = [0.0, 0.0, 0.0]
    print('compile fallback:', type(e).__name__)

In [ ]:
assert len(diffs) == 3
assert all(d < 1e-4 for d in diffs)
print('STABLE-shape correctness verified (reuse economics in chapter)')

## What we earned

Correct numbers prove only correctness. Capture (graphs/breaks), stability (guards, recompiles, cache limit), and amortized cost decide whether compilation helped — in order RUNS → CORRECT → CAPTURED → STABLE → WORTH IT.

Chapter 14 asks whether any observed change is real at all: is the delta larger than seed noise, and did the measurement itself change?